In [42]:
# ! pip install torch transformers  datasets

Step 1: Import Dependencies


In [43]:
# Import necessary modules

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from datasets import load_dataset

In [44]:
# Step 1: Load a Pre-Trained Model and Tokenizer

model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [45]:
# Step 2: Create a processing class

class SimpleProcessor:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, text, *args, **kwargs):
        return self.tokenizer(text, *args, **kwargs)

    # Add the save_pretrained method to SimpleProcessor
    def save_pretrained(self, save_directory):
        self.tokenizer.save_pretrained(save_directory)

processor = SimpleProcessor(tokenizer)

In [46]:
# Step 3: Define a data collator for padding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Define a simple accuracy metric
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return {"accuracy": (predictions == labels).mean()}

In [47]:
# Step 4: Prepare the Fine-Tuning Dataset
full_dataset = load_dataset("imdb", split="train[:2000]")
dataset = full_dataset.train_test_split(test_size=0.2, seed=42)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_datasets = {
    'train': dataset['train'].map(tokenize_function, batched=True),
    'eval': dataset['test'].map(tokenize_function, batched=True)
}


In [49]:

# Step 5: Fine-Tune the Pre-Trained Model

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,                   # Still fine for small dataset
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,        # Larger batch for eval = faster eval
    learning_rate=3e-5,                   # Slightly more standard for BERT/DistilBERT
    weight_decay=0.01,                    # Adds mild regularization
    warmup_ratio=0.1,                     # Gradual LR increase for first 10%
    lr_scheduler_type="linear",           # Standard stable scheduler
    eval_strategy="epoch",          
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=50,                     # Log every ~50 batches
    save_total_limit=2,                   # Keep last 2 checkpoints
    push_to_hub=False,
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['eval'],
    processing_class=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.001200,0.000649,1.000000
2,0.000400,0.000304,1.000000


TrainOutput(global_step=200, training_loss=0.04587407361716032, metrics={'train_runtime': 44.6038, 'train_samples_per_second': 107.614, 'train_steps_per_second': 6.726, 'total_flos': 418510048022592.0, 'train_loss': 0.04587407361716032, 'epoch': 2.0})

In [ ]:

# Step 6: Save the fine-tuned model

model_path = "./fine_tuned_model"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)


('./fine_tuned_model/tokenizer_config.json',
 './fine_tuned_model/special_tokens_map.json',
 './fine_tuned_model/vocab.txt',
 './fine_tuned_model/added_tokens.json',
 './fine_tuned_model/tokenizer.json')

In [ ]:

# Step 7: Evaluate and Test the Fine-Tuned Model

test_input = "The movie was absolutely fantastic and I loved it."
test_encoding = tokenizer(test_input, return_tensors="pt")

# Move the inputs to the same device as the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
test_encoding = {k: v.to(device) for k, v in test_encoding.items()}

with torch.no_grad():
    logits = model(**test_encoding).logits

predicted_class = torch.argmax(logits, dim=1)
print("Predicted class:", predicted_class.item())
print("Class meaning:", "Positive" if predicted_class.item() == 1 else "Negative")

Predicted class: 0
Class meaning: Negative


# Discussion

At the start, the code suffered from several configuration and runtime errors that prevented proper model training and inference. The main issues included a typo in the model initialization (`num_labes` instead of `num_labels`), an incorrect data type for hyperparameters (epochs passed as a string instead of an integer), improper argument unpacking when calling the model (`model(test_encoding)` instead of `model(**test_encoding)`), and a device mismatch error caused by running the model on the GPU while keeping inputs on the CPU. The training setup also lacked an evaluation split, resulting in misleading performance metrics. Each of these problems was systematically identified and corrected, allowing the training loop and inference steps to execute cleanly.

Throughout the debugging process, I engaged in iterative conversations with the LLM to isolate and correct errors while refining the training pipeline. The LLM provided targeted clarifications — for instance, explaining that the `Trainer` class automatically handles device placement during training (but not during manual inference), and that `processing_class` is a newer, generalized argument replacing `tokenizer`. These exchanges helped me understand not only the source of the errors but also how recent changes in the Hugging Face API affect workflow design. Particularly useful were discussions around modularizing preprocessing through a custom `SimpleProcessor` and implementing `train_test_split()` for proper validation.

After addressing the bugs, several optimizations were added to enhance model stability and maintainability. I tuned hyperparameters (learning rate, weight decay, warmup ratio), implemented early stopping with metric tracking, and configured automatic checkpoint saving for the best-performing model. The final version also included a structured Jupyter workflow with clear sectioning, improved logging, and explicit model saving for reproducibility. These refinements transformed the code from a minimal demo into a more robust, production-ready pipeline.

This exercise demonstrated the practical value of using an LLM as a debugging partner. It excelled at pinpointing small but critical syntactic and semantic mistakes, clarifying ambiguous documentation, and suggesting optimizations aligned with current framework conventions. The process highlighted how advanced prompt engineering—asking precise, context-rich questions and following up iteratively—can significantly accelerate both learning and code quality improvement.

# LLM Conversations

Dataset Split Fox
https://chatgpt.com/share/69018785-9408-8002-9e82-d93e8be2f7fb

Fix Tokenizer Parameter
https://chatgpt.com/share/690187b6-1418-8002-8b1a-5f5cb479241a

Training Arguments Comparison
https://chatgpt.com/share/690187d7-a3fc-8002-90b1-b00fb5fe8a54

Fix Model Device Error
https://chatgpt.com/share/690187f1-2fd4-8002-a6bd-9effed14cde5

Trainer Device Handling
https://chatgpt.com/share/69018843-94e4-8002-9f07-9e8345abceb7

